## Configuration

In [0]:
project_identifier = "dar065_2"
project_catalog = "5_projects"
cohort_catalog = "6_mgmt"
cohort_schema = "cohorts"
source_catalog = "4_prod"

project_schema = f"{project_catalog}.{project_identifier}"
cohort_view = f"{cohort_catalog}.{cohort_schema}.{project_identifier}"

cohort_start_date = "2016-07-14"
cohort_end_date = "2026-07-24"

# Columns must pass both thresholds. Untagged columns are excluded unless explicitly included.
max_ig_risk = 3
max_ig_severity = 2

# Always exclude these columns regardless of source tags.
columns_to_exclude = [
    "ADC_UPDT",
]

# Optional reviewed overrides use the DAR template dictionary format.
# AnonymizedText already passes the configured IG thresholds.
columns_to_include = {}

unfiltered_tables = []

# Core structured sources exposed as cohort-filtered project views.
rde_tables = [
    "rde_patient_demographics",
    "rde_all_diagnosis",
    "rde_all_problems",
    "rde_all_procedures",
    "rde_apc_diagnosis",
    "rde_op_diagnosis",
    "rde_cds_apc",
    "rde_cds_opa",
    "rde_encounter",
    "rde_measurements",
    "rde_pharmacyorders",
    "rde_medadmin",
    "rde_blobdataset",
]

map_tables = [
    "map_encounter",
    "map_patient_journey",
]

omop_tables = []
mill_tables = []

# Supporting sources that require bespoke linkage and must not be exposed directly.
side_sources = {
    "referral_detail": [
        "4_prod.raw.cds_op_all",
    ],
    "scheduling_and_listing": [
        "4_prod.raw.mill_sch_appt",
        "4_prod.raw.mill_sch_event",
        "4_prod.raw.mill_surgical_case",
        "4_prod.raw.mill_surg_case_procedure",
        "4_prod.rde.code_value",
    ],
    "supplementary_iweb_only": [
        "4_prod.iweb.reg_noncoronary",
        "4_prod.rde.rde_aliases",
    ],
    "deprivation": [
        "4_prod.bronze.map_address",
    ],
}

## Cohort

In [0]:
# Broad adult denominator: first in-period I47-I49 or clinically named arrhythmia.
# Ablation is identified from the approved RDE OPCS code set.
cohort_sql = f"""
CREATE OR REPLACE VIEW {cohort_view} AS
WITH arrhythmia_events AS (
    SELECT
        Person_ID AS PERSON_ID,
        CAST(Diagnosis_date AS DATE) AS event_date
    FROM {source_catalog}.rde.rde_all_diagnosis
    WHERE CAST(Diagnosis_date AS DATE) BETWEEN DATE '{cohort_start_date}' AND DATE '{cohort_end_date}'
      AND (
          regexp_replace(upper(coalesce(Diagnosis_code, '')), '[^A-Z0-9]', '') RLIKE '^I4[789]'
          OR lower(coalesce(Code_text, '')) RLIKE
             'atrial fibrillation|atrial flutter|supraventricular tachycardia|ventricular tachycardia|cardiac arrhythmia'
      )

    UNION ALL

    SELECT
        Person_ID AS PERSON_ID,
        CAST(Onset_date AS DATE) AS event_date
    FROM {source_catalog}.rde.rde_all_problems
    WHERE CAST(Onset_date AS DATE) BETWEEN DATE '{cohort_start_date}' AND DATE '{cohort_end_date}'
      AND (
          regexp_replace(upper(coalesce(Problem_code, '')), '[^A-Z0-9]', '') RLIKE '^I4[789]'
          OR lower(coalesce(Code_text, '')) RLIKE
             'atrial fibrillation|atrial flutter|supraventricular tachycardia|ventricular tachycardia|cardiac arrhythmia'
      )
),
index_dates AS (
    SELECT PERSON_ID, MIN(event_date) AS index_date
    FROM arrhythmia_events
    WHERE PERSON_ID IS NOT NULL AND event_date IS NOT NULL
    GROUP BY PERSON_ID
),
ablation_people AS (
    SELECT DISTINCT Person_ID AS PERSON_ID
    FROM {source_catalog}.rde.rde_all_procedures
    WHERE CAST(Procedure_date AS DATE) BETWEEN DATE '{cohort_start_date}' AND DATE '{cohort_end_date}'
      AND regexp_replace(upper(coalesce(Procedure_code, '')), '[^A-Z0-9]', '') IN
          ('K571', 'K572', 'K574', 'K575', 'K576', 'K621', 'K622', 'K623')
)
SELECT
    i.PERSON_ID,
    CASE WHEN a.PERSON_ID IS NOT NULL THEN 'ABLATION' ELSE 'NON_ABLATION' END AS SUBCOHORT
FROM index_dates i
INNER JOIN {source_catalog}.rde.rde_patient_demographics d
    ON i.PERSON_ID = d.PERSON_ID
LEFT JOIN ablation_people a
    ON i.PERSON_ID = a.PERSON_ID
WHERE coalesce(
          CAST(floor(months_between(i.index_date, d.Date_of_Birth) / 12) AS INT),
          year(i.index_date) - d.Year_of_Birth
      ) >= 18
"""

spark.sql(cohort_sql)
print(f"Created cohort view: {cohort_view}")

## Schema Setup

In [0]:
spark.sql("USE CATALOG 5_projects")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS 5_projects.{project_identifier}")

existing_views_df = spark.sql(f"SHOW VIEWS IN 5_projects.{project_identifier}")
existing_views = {row.viewName for row in existing_views_df.collect()}
for view_name in existing_views:
    spark.sql(f"DROP VIEW IF EXISTS 5_projects.{project_identifier}.{view_name}")
    print(f"Dropped view: 5_projects.{project_identifier}.{view_name}")

existing_tables_df = spark.sql(f"SHOW TABLES IN 5_projects.{project_identifier}")
for row in existing_tables_df.collect():
    if row.tableName not in existing_views and row.tableName != project_identifier:
        spark.sql(f"DROP TABLE IF EXISTS 5_projects.{project_identifier}.{row.tableName}")
        print(f"Dropped table: 5_projects.{project_identifier}.{row.tableName}")

## Helper Functions

In [0]:
def quote_identifier(value):
    tick = chr(96)
    return tick + value.replace(tick, tick + tick) + tick


def get_safe_columns(catalog, schema, table):
    """Return tagged columns within the template IG thresholds."""
    full_path = f"{catalog}.{schema}.{table}"
    all_columns = spark.table(full_path).columns

    safe_df = spark.sql(f"""
        SELECT r.column_name
        FROM (
            SELECT column_name, CAST(tag_value AS INT) AS risk_val
            FROM {catalog}.information_schema.column_tags
            WHERE schema_name = '{schema}'
              AND table_name = '{table}'
              AND tag_name = 'ig_risk'
        ) r
        JOIN (
            SELECT column_name, CAST(tag_value AS INT) AS sev_val
            FROM {catalog}.information_schema.column_tags
            WHERE schema_name = '{schema}'
              AND table_name = '{table}'
              AND tag_name = 'ig_severity'
        ) s ON r.column_name = s.column_name
        WHERE r.risk_val <= {max_ig_risk}
          AND s.sev_val <= {max_ig_severity}
    """)
    safe_columns = {row.column_name for row in safe_df.collect()}

    safe_columns |= set(columns_to_include.get(full_path, []))
    safe_columns |= set(columns_to_include.get("*", []))

    excluded_lower = {column.lower() for column in columns_to_exclude}
    return [
        column
        for column in all_columns
        if column in safe_columns and column.lower() not in excluded_lower
    ]


def find_person_id_column(catalog, schema, table):
    """Find the pseudonymous person identifier used to apply the cohort."""
    columns = spark.table(f"{catalog}.{schema}.{table}").columns
    candidates = [
        "PERSON_ID",
        "person_id",
        "Person_ID",
        "PERSONID",
        "PersonID",
        "participant_id",
    ]
    for candidate in candidates:
        if candidate in columns:
            return candidate
    for column in columns:
        if "person" in column.lower() and "id" in column.lower():
            return column
    return None


def create_cohort_filtered_view(catalog, schema, table, project_id, column_list=None, alias=None):
    """Create the standard template-style project view."""
    full_path = f"{catalog}.{schema}.{table}"
    view_name = alias or table

    if column_list is None:
        column_list = get_safe_columns(catalog, schema, table)
        if not column_list:
            print(f"WARNING: No columns passed IG filtering for {full_path}. Skipping.")
            return False

    person_id_column = find_person_id_column(catalog, schema, table)
    if column_list == ["*"]:
        columns_sql = "t.*"
    else:
        columns_sql = ", ".join(
            f"t.{quote_identifier(column)}" for column in column_list
        )

    if person_id_column:
        view_sql = f"""
        CREATE OR REPLACE VIEW 5_projects.{project_id}.{view_name} AS
        SELECT {columns_sql}
        FROM {full_path} t
        INNER JOIN 6_mgmt.cohorts.{project_id} c
          ON t.{quote_identifier(person_id_column)} = c.PERSON_ID
        """
    else:
        print(f"Note: no person ID column in {full_path}; view is not cohort-filtered.")
        view_sql = f"""
        CREATE OR REPLACE VIEW 5_projects.{project_id}.{view_name} AS
        SELECT {columns_sql}
        FROM {full_path} t
        """

    spark.sql(view_sql)
    print(f"Created view: 5_projects.{project_id}.{view_name}")
    return True

## Create Views

In [0]:
created = []
failed = []

source_groups = [
    ("rde", rde_tables),
    ("bronze", map_tables),
    ("omop", omop_tables),
    ("raw", mill_tables),
]

for source_schema, table_list in source_groups:
    for table in table_list:
        try:
            if create_cohort_filtered_view(source_catalog, source_schema, table, project_identifier):
                created.append(f"{source_schema}.{table}")
        except Exception as exc:
            failed.append((f"{source_schema}.{table}", str(exc)))
            print(f"ERROR: {source_schema}.{table}: {exc}")

## Bespoke Side Views

In [0]:


# IMD from the canonical address map.
# Publish one row per person without postcode, address, LSOA, UPRN or coordinates.
spark.sql(f"""
CREATE OR REPLACE VIEW {project_schema}.deprivation AS
WITH ranked AS (
    SELECT
        CAST(a.PARENT_ENTITY_ID AS BIGINT) AS PERSON_ID,
        try_cast(a.IMD_Decile AS INT) AS imd_decile,
        try_cast(a.IMD_Quintile AS INT) AS imd_quintile,
        a.match_algorithm,
        a.match_confidence,
        a.match_quality,
        row_number() OVER (
            PARTITION BY a.PARENT_ENTITY_ID
            ORDER BY
                CASE
                    WHEN a.ACTIVE_IND = 1
                     AND (
                         a.BEG_EFFECTIVE_DT_TM IS NULL
                         OR a.BEG_EFFECTIVE_DT_TM <= TIMESTAMP '{cohort_end_date} 23:59:59'
                     )
                     AND (
                         a.END_EFFECTIVE_DT_TM IS NULL
                         OR a.END_EFFECTIVE_DT_TM >= TIMESTAMP '{cohort_end_date} 00:00:00'
                     )
                    THEN 0 ELSE 1
                END,
                a.BEG_EFFECTIVE_DT_TM DESC NULLS LAST,
                a.ADDRESS_ID DESC
        ) AS row_number
    FROM {source_catalog}.bronze.map_address a
    INNER JOIN {cohort_view} c
        ON CAST(a.PARENT_ENTITY_ID AS BIGINT) = c.PERSON_ID
    WHERE upper(coalesce(a.PARENT_ENTITY_NAME, '')) = 'PERSON'
)
SELECT
    PERSON_ID,
    imd_decile,
    imd_quintile,
    match_algorithm,
    match_confidence,
    match_quality
FROM ranked
WHERE row_number = 1
""")
created.append("bespoke.deprivation")

## Schema Documentation View

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {project_schema}.schema AS
SELECT
    table_name,
    column_name,
    data_type,
    coalesce(comment, '') AS column_comment
FROM {project_catalog}.information_schema.columns
WHERE table_catalog = '{project_catalog}'
  AND table_schema = '{project_identifier}'
  AND table_name <> 'schema'
ORDER BY table_name, ordinal_position
""")

if failed:
    raise RuntimeError(f"Source views failed: {failed}")

print(f"Project setup complete: {cohort_view} -> {project_schema}")